# Customer Lifetime Value (CLV): Monte Carlo Portfolio Analysis
## Probabilistic Revenue Forecasting for a Synthetic Telco Customer Base

---

> **Project:** CLV Monte Carlo Simulation  
> **Segments:** High-Value · Standard · Churn-Risk  
> **Method:** Monte Carlo with Weibull survival and Lognormal spend models  
> **Tools:** Python · NumPy · Plotly · Pandas

---

*This notebook is the capstone of a three-part analysis series. It synthesises data exploration (Notebook 01) and distribution fitting (Notebook 02) into a complete portfolio CLV model — covering single-customer uncertainty, portfolio-level distributions, risk metrics, and sensitivity analysis.*

---
## 1. Introduction

### What is Customer Lifetime Value?

Customer Lifetime Value (CLV) is the **total discounted revenue** a business expects to earn from a single customer over the entire duration of their relationship. It is arguably the single most strategically important metric in subscription-based businesses:

> *"CLV answers not just what a customer is worth today, but what they will be worth — in today's money — over the life of their entire relationship with your brand."*

### Why CLV Matters for Telcos

South African telcos operate in a capital-intensive environment where acquiring new subscribers is expensive and margins are thin. CLV provides the financial foundation for four critical decisions:

| Business Decision | How CLV Answers It |
|---|---|
| **Customer Acquisition Cost** | Don't spend more to acquire a customer than their CLV |
| **Retention Investment** | Target retention budget at high-CLV customers showing churn signals |
| **Segment Strategy** | Identify which product tiers to grow aggressively vs. rationalise |
| **Pricing** | Price plans so that expected CLV comfortably exceeds cost-to-serve + acquire |

Without CLV, businesses fly blind — spending equal acquisition budget on customers with wildly different revenue potential, or triggering retention interventions too late for customers who were never going to stay.

### Why a Point Estimate is Dangerous

A single "expected CLV" hides enormous uncertainty. Consider two customers with identical point-estimate CLV of R 2,000:

- **Customer A:** 90% of simulated CLV paths fall between R 1,600 – R 2,400 *(very predictable, low risk)*
- **Customer B:** 90% of simulated CLV paths fall between R 200 – R 5,000 *(highly uncertain, high risk)*

Treating them identically is a costly mistake. The business might over-invest in Customer B who turns out to be a fast churner, or fail to protect Customer A who is quietly, reliably valuable.

**Monte Carlo simulation** solves this by drawing thousands of possible futures for each customer — sampling from empirically calibrated churn timing and spending distributions — and returning a full probability distribution rather than a single number. This notebook demonstrates that framework end-to-end.

### What This Notebook Covers

| Section | You Will Learn |
|---|---|
| **§2 Data** | The synthetic portfolio: 10,000 customers across 3 behavioural segments |
| **§3 Framework** | The CLV formula, Weibull survival model, and Monte Carlo integration |
| **§4 Single Customer** | What uncertainty looks like for one individual customer |
| **§5 Portfolio** | How individual uncertainty aggregates across a 1,000-customer portfolio |
| **§6 Risk Metrics** | Segment medians, percentiles, at-risk customers, and optimal CAC |
| **§7 Sensitivity** | Which input assumptions drive the most CLV uncertainty? |
| **§8 Implications** | What the numbers mean for acquisition budgets and retention strategy |

In [ ]:
import sys
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Make the src/ module importable from the notebooks/ directory
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from simulation import CLVSimulation, MCConfig
from risk_metrics import RiskMetrics, SensitivityAnalyzer

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Consistent colour palette (matches Notebooks 01 & 02) ────────────────────
SEGMENT_COLORS = {
    'high_value': '#2E86AB',   # steel blue
    'standard':   '#A23B72',   # plum
    'churn_risk': '#F18F01',   # amber
}

SEGMENT_LABELS = {
    'high_value': 'High Value',
    'standard':   'Standard',
    'churn_risk': 'Churn Risk',
}

SEGMENTS = ['high_value', 'standard', 'churn_risk']
DATA_PATH = '../reports/synthetic_customers.csv'

print('Libraries loaded successfully.')
print(f'  NumPy   {np.__version__}')
print(f'  Pandas  {pd.__version__}')

---
## 2. Data: The Customer Portfolio

Our synthetic dataset models a South African telco's customer base with three behavioural tiers. Each customer record contains four key fields:

| Column | Description | Generating Distribution |
|---|---|---|
| `monthly_spending` | Spend per active month (R) | Gamma(α, β) — strictly positive, right-skewed |
| `months_to_churn` | Months until the customer leaves | Weibull(k, λ) — time-to-event with shape-dependent hazard |
| `segment` | Behavioural tier | Categorical: `high_value` / `standard` / `churn_risk` |
| `contract_type` | Contract structure | Categorical: month-to-month / annual / multi-year |

The distribution parameters were calibrated by maximum likelihood estimation in Notebook 02 and are stored in `reports/fitted_params.json`.

In [ ]:
customers = pd.read_csv(DATA_PATH)

print(f'Dataset: {len(customers):,} customers x {customers.shape[1]} features')
print(f'Segments: {dict(customers["segment"].value_counts().sort_index())}\n')

# Display a clean random sample
sample_display = (
    customers
    .sample(8, random_state=SEED)
    .reset_index(drop=True)
    [['customer_id', 'segment', 'monthly_spending', 'months_to_churn', 'contract_type']]
)

sample_display.style \
    .format({'monthly_spending': 'R{:,.2f}', 'months_to_churn': '{:.1f} mo'}) \
    .set_caption('Table 1: Random sample of 8 customers') \
    .background_gradient(subset=['monthly_spending'], cmap='Blues', low=0.2) \
    .background_gradient(subset=['months_to_churn'], cmap='Oranges', low=0.2)

In [ ]:
# ── Per-segment summary statistics ───────────────────────────────────────────
seg_summary = (
    customers.groupby('segment').agg(
        n_customers  =('customer_id',      'count'),
        pct_of_base  =('customer_id',      lambda x: f'{100 * len(x) / len(customers):.1f}%'),
        avg_spend    =('monthly_spending', 'mean'),
        median_spend =('monthly_spending', 'median'),
        std_spend    =('monthly_spending', 'std'),
        avg_churn_mo =('months_to_churn',  'mean'),
        med_churn_mo =('months_to_churn',  'median'),
    )
    .reindex(SEGMENTS)
    .round(2)
)

# ── Interactive Plotly table ──────────────────────────────────────────────────
header_vals = ['Segment', 'N', '% Base', 'Avg Spend', 'Median Spend',
               'Std Spend', 'Avg Lifetime', 'Median Lifetime']

cell_vals = [
    [SEGMENT_LABELS[s] for s in seg_summary.index],
    [f'{v:,}' for v in seg_summary['n_customers']],
    seg_summary['pct_of_base'].tolist(),
    [f'R{v:,.2f}' for v in seg_summary['avg_spend']],
    [f'R{v:,.2f}' for v in seg_summary['median_spend']],
    [f'R{v:,.2f}' for v in seg_summary['std_spend']],
    [f'{v:.1f} mo' for v in seg_summary['avg_churn_mo']],
    [f'{v:.1f} mo' for v in seg_summary['med_churn_mo']],
]

row_colors = ['#EAF4FB', '#FDFEFE', '#FEF9F0']
fig_tbl = go.Figure(data=[go.Table(
    header=dict(
        values=header_vals,
        fill_color='#2E3440',
        font=dict(color='white', size=12, family='Arial'),
        align='center',
        height=36,
    ),
    cells=dict(
        values=cell_vals,
        fill_color=[[row_colors[i] for i in range(3)] for _ in header_vals],
        font=dict(size=12),
        align=['left', 'center', 'center', 'right', 'right', 'right', 'center', 'center'],
        height=32,
    ),
)])
fig_tbl.update_layout(
    title='Table 2: Segment Breakdown — Spend and Lifetime Characteristics',
    height=230,
    margin=dict(l=20, r=20, t=65, b=20),
)
fig_tbl.show()

---
## 3. Mathematical Framework

### 3.1 The CLV Formula

Customer Lifetime Value is the **net present value (NPV)** of all future cash flows from a customer, accumulated over their active lifetime and discounted back to today's money:

$$\text{CLV} = \sum_{t=1}^{T} \frac{m_t \cdot \mathbb{1}[t \leq \tau]}{(1+r)^t}$$

where:

| Symbol | Meaning |
|--------|--------|
| $m_t$ | Monthly spend in month $t$ |
| $\tau$ | The customer's churn month (sampled from their churn distribution) |
| $\mathbb{1}[t \leq \tau]$ | Indicator: 1 if the customer is still active in month $t$, else 0 |
| $r$ | Monthly discount rate (derived from the annual rate) |
| $T$ | Planning horizon — 60 months (5 years) in this model |

### 3.2 NPV Discounting

A rand received today is worth more than a rand received next year, because today's rand can be invested. The monthly discount rate is derived from the annual WACC via compound interest:

$$r_{\text{monthly}} = (1 + r_{\text{annual}})^{1/12} - 1$$

At 12% p.a., this gives $r_{\text{monthly}} \approx 0.949\%$ per month — not the naive $12\% \div 12 = 1\%$, which overstates the discount and undervalues near-term cash flows.

### 3.3 The Spend Model (Lognormal)

Monthly spending $m_t$ is modelled as **lognormal** — strictly positive and right-skewed, matching observed customer spend patterns:

$$m_t \sim \text{LogNormal}(\mu,\; \sigma = 0.10)$$

The location parameter $\mu$ is chosen so that the expected value equals the customer's historical average spend $\bar{m}$:

$$\mu = \ln(\bar{m}) - \frac{\sigma^2}{2} \quad \Rightarrow \quad \mathbb{E}[m_t] = \bar{m}$$

This adds realistic ±10% month-to-month variability (coefficient of variation = 10%) while preserving the correct expected spend.

### 3.4 The Survival Model (Weibull)

The churn month $\tau$ is drawn from a Weibull-parameterised lifetime distribution. The underlying **Weibull survival function** — the probability a customer survives beyond time $t$ — is:

$$S(t) = e^{-(t/\lambda)^k}$$

where $\lambda$ (scale) is the characteristic lifetime and $k$ (shape) controls the hazard behaviour:

| Shape $k$ | Hazard Rate | Telco Interpretation |
|-----------|-------------|---------------------|
| $k < 1$ | Decreasing | **Infant mortality** — churn risk is highest right after acquisition, then falls. Our `churn_risk` segment ($k \approx 0.83$). |
| $k = 1$ | Constant | **Memoryless** — churn probability is the same at any tenure. Our `standard` segment ($k \approx 1.01$). |
| $k > 1$ | Increasing | **Wear-out** — long-tenure customers become progressively more likely to churn. Our `high_value` segment ($k \approx 1.18$). |

### 3.5 Monte Carlo Integration

Rather than computing the CLV expectation analytically (intractable due to the product of two random variables), we use **Monte Carlo integration**: draw $N$ independent lifetime paths and average the results:

$$\widehat{\text{CLV}} = \frac{1}{N} \sum_{i=1}^{N} \text{CLV}_i$$

The key insight is that we examine the **entire distribution** $\{\text{CLV}_1, \ldots, \text{CLV}_N\}$ — not just its mean — giving us a complete picture of uncertainty. With $N = 1{,}000$ paths, the standard error of the median estimate is well under 1%.

---
## 4. Single-Customer Simulation

Before scaling to the full portfolio, let's look closely at **one representative customer per segment** — the customer closest to the segment's median spending level. We run 1,000 Monte Carlo paths for each, producing a full CLV distribution.

This is where the power of the probabilistic approach becomes tangible: instead of a single number, we see the full range of outcomes the business might realistically experience with this customer.

In [ ]:
# ── Pick the customer closest to the segment median spend ─────────────────────
rep_customers = {}
print('Representative customers selected:\n')

for seg in SEGMENTS:
    seg_df = customers[customers['segment'] == seg].copy()
    median_spend = seg_df['monthly_spending'].median()
    rep_idx = (seg_df['monthly_spending'] - median_spend).abs().idxmin()
    rep_customers[seg] = seg_df.loc[rep_idx]
    c = rep_customers[seg]
    print(
        f'  {SEGMENT_LABELS[seg]:>12} | ID: {int(c["customer_id"]):>5} | '
        f'Spend: R{c["monthly_spending"]:>7.2f}/mo | '
        f'Expected lifetime: {c["months_to_churn"]:.1f} mo'
    )

# ── Run 1,000 Monte Carlo paths per representative customer ───────────────────
config_single = MCConfig(
    n_simulations=1_000,
    planning_horizon_months=60,
    discount_rate_annual=0.12,
    seed=SEED,
)
sim_engine = CLVSimulation(customers=customers, config=config_single)

clv_distributions = {}
print('\nMonte Carlo CLV distributions (1,000 paths each):\n')
print(f'  {"Segment":>12} | {"P5":>8} | {"P25":>8} | {"Median":>9} | {"P75":>8} | {"P95":>8} | {"Std Dev":>8}')
print('  ' + '-' * 72)

for seg, customer in rep_customers.items():
    paths = sim_engine.simulate_customer_path(customer, n_sims=1_000)
    clv_distributions[seg] = paths
    p5, p25, p50, p75, p95 = np.percentile(paths, [5, 25, 50, 75, 95])
    print(
        f'  {SEGMENT_LABELS[seg]:>12} | '
        f'R{p5:>7,.0f} | R{p25:>7,.0f} | R{p50:>8,.0f} | '
        f'R{p75:>7,.0f} | R{p95:>7,.0f} | R{paths.std():>7,.0f}'
    )

In [ ]:
# ── Build subplot titles with customer details ────────────────────────────────
subtitles = []
for s in SEGMENTS:
    cid    = int(rep_customers[s]['customer_id'])
    spend  = rep_customers[s]['monthly_spending']
    tenure = rep_customers[s]['months_to_churn']
    subtitles.append(
        f'{SEGMENT_LABELS[s]}<br>'
        f'<sup>ID {cid} · R{spend:.0f}/mo · {tenure:.0f} mo expected tenure</sup>'
    )

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=subtitles,
    horizontal_spacing=0.07,
)

for col_idx, seg in enumerate(SEGMENTS, 1):
    paths = clv_distributions[seg]
    p5, p50, p95 = np.percentile(paths, [5, 50, 95])

    fig.add_trace(
        go.Histogram(
            x=paths,
            marker_color=SEGMENT_COLORS[seg],
            opacity=0.82,
            nbinsx=50,
            name=SEGMENT_LABELS[seg],
            showlegend=False,
            hovertemplate='CLV: R%{x:,.0f}<br>Count: %{y}<extra></extra>',
        ),
        row=1, col=col_idx,
    )

    # Percentile reference lines: P5 = red dotted, P50 = black solid, P95 = green dotted
    for val, color, dash in [
        (p5,  '#C0392B', 'dot'),
        (p50, '#1A252F', 'solid'),
        (p95, '#1E8449', 'dot'),
    ]:
        fig.add_vline(
            x=val,
            line_color=color, line_dash=dash, line_width=2,
            row=1, col=col_idx,
        )

fig.update_xaxes(title_text='CLV (R)', tickformat=',.0f')
fig.update_yaxes(title_text='Simulated Path Count', row=1, col=1)
fig.update_layout(
    title=dict(
        text=(
            'Figure 1: CLV Distribution for a Representative Customer per Segment '
            '(1,000 Simulated Paths)<br>'
            '<sup>Red dotted = P5 · Black solid = P50 median · Green dotted = P95</sup>'
        ),
        x=0.5,
        font=dict(size=14),
    ),
    template='plotly_white',
    height=460,
    margin=dict(t=120, b=60, l=70, r=30),
)
fig.show()

### What the Spread Tells Us

The histograms make three things immediately visible:

**1. Uncertainty scales with value.** The High Value customer's CLV spans a much wider absolute range than the Churn Risk customer's. But as a *fraction* of the median, all three segments show similar relative uncertainty — roughly ±20-30% coefficient of variation. This is driven by the ±10% monthly spend noise and Gaussian churn timing perturbation built into the simulation.

**2. The shape reveals the dominant risk.** The Churn Risk distribution is heavily left-skewed: the P5 is close to zero (the customer may churn in month 1 or 2), while the P95 is much further from the median than the P5 is in absolute terms. For the High Value distribution, the shape is more symmetric, reflecting a longer, more stable lifetime.

**3. The 90% confidence interval is actionable.** The region from P5 to P95 defines the range of outcomes the business should plan for. For a High Value customer, the business can be confident that CLV will fall in that band 90% of the time. Any acquisition investment that only pays off if CLV exceeds the P95 is a speculative bet — not a sound commercial decision.

> **Key takeaway:** A single point estimate ("this customer is worth R X") discards all of this information. The full distribution is what matters for risk-adjusted decisions about acquisition spending, retention investment, and churn intervention timing.

---
## 5. Portfolio Simulation

Individual-customer uncertainty is interesting but incomplete. In practice, the business makes decisions at the **portfolio level** — it cannot predict which specific customers will churn early, but it can predict the aggregate distribution of outcomes across all customers.

Here we sample **1,000 customers** from the full dataset, run 500 Monte Carlo paths per customer, and examine how the CLV distributions differ across segments at scale.

> **Why 1,000 customers and 500 paths?** This is a deliberate trade-off between accuracy and notebook execution speed. For production scoring, you would run all 10,000 customers × 10,000+ paths. The distributions converge well before these limits — the segment-level medians are stable to within 1-2% with these settings.

In [ ]:
PORTFOLIO_N = 1_000  # customers sampled for notebook speed

portfolio_sample = customers.sample(n=PORTFOLIO_N, random_state=SEED).reset_index(drop=True)

config_portfolio = MCConfig(
    n_simulations=500,          # 500 paths per customer
    planning_horizon_months=60,
    discount_rate_annual=0.12,
    seed=SEED,
)

print(
    f'Running simulation: {PORTFOLIO_N:,} customers '
    f'x {config_portfolio.n_simulations:,} paths '
    f'x {config_portfolio.planning_horizon_months} months\n'
    f'This typically takes 30-90 seconds...\n'
)

sim_portfolio = CLVSimulation(customers=portfolio_sample, config=config_portfolio)
portfolio_results = sim_portfolio.run_full_simulation()

print(f'Simulation complete: {len(portfolio_results):,} customer CLV distributions.')
print()
for seg in SEGMENTS:
    seg_data = portfolio_results[portfolio_results['segment'] == seg]
    med = seg_data['clv_p50'].median()
    std = seg_data['clv_p50'].std()
    print(f'  {SEGMENT_LABELS[seg]:>12}: median CLV = R{med:>8,.2f}  |  std = R{std:>8,.2f}')

In [ ]:
fig_port = go.Figure()

for seg in SEGMENTS:
    seg_clv = portfolio_results[portfolio_results['segment'] == seg]['clv_p50']
    seg_label = SEGMENT_LABELS[seg]
    fig_port.add_trace(go.Histogram(
        x=seg_clv,
        name=seg_label,
        marker_color=SEGMENT_COLORS[seg],
        opacity=0.62,
        nbinsx=55,
        histnorm='probability density',
        hovertemplate=f'<b>{seg_label}</b><br>CLV: R%{{x:,.0f}}<br>Density: %{{y:.5f}}<extra></extra>',
    ))

# Vertical dashed lines at segment medians
for seg in SEGMENTS:
    seg_median = portfolio_results[portfolio_results['segment'] == seg]['clv_p50'].median()
    label_text = f'{SEGMENT_LABELS[seg]}: R{seg_median:,.0f}'
    fig_port.add_vline(
        x=seg_median,
        line_color=SEGMENT_COLORS[seg],
        line_dash='dash',
        line_width=2.5,
        annotation_text=label_text,
        annotation_position='top right',
        annotation_font_size=10,
        annotation_font_color=SEGMENT_COLORS[seg],
    )

fig_port.update_layout(
    barmode='overlay',
    title=dict(
        text=(
            'Figure 2: Portfolio CLV Distribution by Segment<br>'
            '<sup>Median CLV (P50) per customer across '
            f'{PORTFOLIO_N:,}-customer portfolio · '
            'Dashed lines = segment medians</sup>'
        ),
        x=0.5,
        font=dict(size=14),
    ),
    xaxis_title='Median CLV per Customer (R)',
    yaxis_title='Probability Density',
    legend=dict(x=0.80, y=0.95, bgcolor='rgba(255,255,255,0.85)', bordercolor='#BDC3C7', borderwidth=1),
    template='plotly_white',
    height=470,
    margin=dict(t=105, b=65),
)
fig_port.show()

---
## 6. Key Risk Metrics

With the portfolio simulation complete, we can now compute the standard risk metrics used to characterise CLV uncertainty. These fall into two categories:

**Distribution metrics** — summarise where the bulk of value sits and how dispersed it is:
- **Median CLV (P50):** The central estimate — 50% of customers are above, 50% below.
- **P5 / P95 CLV:** The pessimistic and optimistic tails — the range in which 90% of customers fall.
- **Std Dev:** The absolute spread of CLV across the segment.

**At-risk metrics** — identify customers who may destroy rather than create value:
- A customer is "at risk" if their **P5 CLV** (the pessimistic tail of *their own* simulation) falls below the portfolio's 25th-percentile floor.
- This identifies customers where the downside scenario is so bad that they represent an acquisition risk.

In [ ]:
rm = RiskMetrics(portfolio_results)

# ── Segment breakdown table ───────────────────────────────────────────────────
breakdown = rm.segment_breakdown().reset_index()

cell_metrics = [
    [SEGMENT_LABELS.get(s, s) for s in breakdown['segment']],
    [f'{v:,}' for v in breakdown['n_customers']],
    [f'R{v:,.2f}' for v in breakdown['median_clv']],
    [f'R{v:,.2f}' for v in breakdown['p5_clv']],
    [f'R{v:,.2f}' for v in breakdown['p95_clv']],
    [f'R{v:,.2f}' for v in breakdown['std_clv']],
]
hdr_metrics = ['Segment', 'Customers', 'Median CLV (P50)', 'P5 CLV (Bear)', 'P95 CLV (Bull)', 'Std Dev']

row_fill = ['#EAF4FB', '#FEF9F0', '#FEF5FB']  # blue, amber, plum tint per segment

fig_m = go.Figure(data=[go.Table(
    header=dict(
        values=hdr_metrics,
        fill_color='#2E3440',
        font=dict(color='white', size=13, family='Arial'),
        align='center',
        height=38,
    ),
    cells=dict(
        values=cell_metrics,
        fill_color=[[row_fill[i] for i in range(len(breakdown))] for _ in hdr_metrics],
        font=dict(size=13),
        align=['left', 'center', 'right', 'right', 'right', 'right'],
        height=34,
    ),
)])
fig_m.update_layout(
    title='Table 3: CLV Risk Metrics by Segment',
    height=240,
    margin=dict(l=20, r=20, t=65, b=20),
)
fig_m.show()

# ── Portfolio-level summary ───────────────────────────────────────────────────
summary  = rm.portfolio_summary()
at_risk  = rm.customers_at_risk(threshold_percentile=25)

print('Portfolio-level statistics (all segments combined):')
print(f'  Median CLV:                  R{summary["median_clv"]:>10,.2f}')
print(f'  Mean CLV:                    R{summary["mean_clv"]:>10,.2f}')
print(f'  P5  CLV (pessimistic tail):  R{summary["clv_5th_pct"]:>10,.2f}')
print(f'  P95 CLV (optimistic tail):   R{summary["clv_95th_pct"]:>10,.2f}')
print(f'  % customers with neg. P5:    {summary["pct_negative_p5"]:>10.1f}%')
print(f'\nAt-risk customers (P5 CLV below 25th-pct floor of R{at_risk["threshold_clv"]:,.2f}):')
print(f'  Count:  {at_risk["n_at_risk"]:>6,} customers  ({at_risk["pct_at_risk"]:.1f}% of portfolio)')

---
## 7. Sensitivity Analysis — The Tornado Chart

### What Is a Sensitivity Analysis?

Even a well-calibrated CLV model depends on assumptions about the future. Two assumptions matter most in our framework:

1. **Discount rate (WACC):** How much we discount future rands to their present value. A higher WACC makes distant cash flows less valuable — compressing CLV. A lower WACC does the opposite.
2. **Churn rate:** Whether customers actually stay as long as we expect. A 20% improvement in retention (customers live 20% longer) is a fundamentally different business outcome than a 20% deterioration.

A **one-at-a-time (OAT) sensitivity analysis** quantifies the business impact of being wrong about each assumption:

1. Run the simulation once with baseline settings — this is the reference point.
2. For each uncertain parameter, re-run with a **low value** and a **high value**, holding everything else fixed.
3. Measure how much the portfolio median CLV shifts (as % of baseline) at each extreme.
4. Sort parameters from **largest swing to smallest** — this produces the tornado chart.

### Why the Tornado Shape Matters

The tornado chart is named for its shape: the parameter with the widest swing sits at the top (the widest part of the funnel), and progressively smaller-impact parameters stack below it.

**Reading the chart:**
- **Long bar** = this assumption matters enormously; focus modelling effort and operational budget here.
- **Short bar** = this assumption can be fixed at its baseline value without much loss of accuracy.
- **Direction** = red bars extend left (negative CLV impact); teal bars extend right (positive CLV impact).

### Parameters Tested

| Parameter | Low Value | High Value | Rationale |
|---|---|---|---|
| `discount_rate_annual` | 10% | 16% | Plausible WACC range for a South African telco |
| `churn_multiplier` | ×0.80 | ×1.20 | 20% better or worse retention than the base forecast |

In [ ]:
SENSITIVITY_N = 300  # smaller sample — 5 simulation runs total, keeps this fast

sensitivity_customers = (
    customers.sample(n=SENSITIVITY_N, random_state=SEED).reset_index(drop=True)
)

config_sensitivity = MCConfig(
    n_simulations=300,
    planning_horizon_months=60,
    discount_rate_annual=0.12,
    seed=SEED,
)

param_ranges = {
    'discount_rate_annual': (0.10, 0.16),  # 10% to 16% annual WACC
    'churn_multiplier':     (0.80, 1.20),  # 20% better or 20% worse retention
}

print('Running OAT sensitivity analysis...')
print(f'  Sample:     {SENSITIVITY_N} customers x {config_sensitivity.n_simulations} paths')
print(f'  Parameters: {list(param_ranges.keys())}')
print(f'  Total runs: 1 baseline + {2 * len(param_ranges)} variations = {1 + 2 * len(param_ranges)}\n')

analyzer = SensitivityAnalyzer(customers=sensitivity_customers, config=config_sensitivity)
sensitivity_df = analyzer.analyze(param_ranges)

print('\nSensitivity results summary:')

sensitivity_df[[
    'parameter', 'low_value', 'high_value',
    'change_low_pct', 'change_high_pct', 'abs_range',
]].style \
    .format({
        'change_low_pct':  '{:+.2f}%',
        'change_high_pct': '{:+.2f}%',
        'abs_range':       '{:.2f} pp',
    }) \
    .bar(subset=['abs_range'], color='#5DADE2') \
    .set_caption('Table 4: OAT Sensitivity Results — impact on median portfolio CLV')

In [ ]:
df_t = sensitivity_df.copy()

# ── Build human-readable y-axis labels ───────────────────────────────────────
dr_row = df_t[df_t['parameter'] == 'discount_rate_annual'].iloc[0]
cm_row = df_t[df_t['parameter'] == 'churn_multiplier'].iloc[0]

param_display = {
    'discount_rate_annual': (
        f'Annual Discount Rate (WACC)<br>'
        f'<sup>Low: {dr_row["low_value"]:.0%}  |  Baseline: 12%  |  High: {dr_row["high_value"]:.0%}</sup>'
    ),
    'churn_multiplier': (
        f'Churn Rate Multiplier<br>'
        f'<sup>Low: x{cm_row["low_value"]:.2f} (better retention)  |  '
        f'High: x{cm_row["high_value"]:.2f} (worse retention)</sup>'
    ),
}

y_labels  = [param_display.get(p, p) for p in df_t['parameter'].tolist()]
low_pcts  = df_t['change_low_pct'].tolist()
high_pcts = df_t['change_high_pct'].tolist()
baseline_clv = df_t['baseline_clv'].iloc[0]

# ── Tornado chart: two overlapping horizontal bar traces ─────────────────────
fig_tornado = go.Figure()

fig_tornado.add_trace(go.Bar(
    y=y_labels,
    x=low_pcts,
    orientation='h',
    name='Low Value Scenario',
    marker=dict(
        color=['#E74C3C' if v < 0 else '#1ABC9C' for v in low_pcts],
        opacity=0.88,
    ),
    text=[f'{v:+.1f}%' for v in low_pcts],
    textposition='outside',
    textfont=dict(size=12, color='#2C3E50'),
))

fig_tornado.add_trace(go.Bar(
    y=y_labels,
    x=high_pcts,
    orientation='h',
    name='High Value Scenario',
    marker=dict(
        color=['#1ABC9C' if v >= 0 else '#E74C3C' for v in high_pcts],
        opacity=0.88,
    ),
    text=[f'{v:+.1f}%' for v in high_pcts],
    textposition='outside',
    textfont=dict(size=12, color='#2C3E50'),
))

fig_tornado.update_layout(
    barmode='overlay',
    title=dict(
        text=(
            f'Figure 3: Tornado Chart — What Drives Median Portfolio CLV?<br>'
            f'<sup>Baseline median CLV: R{baseline_clv:,.2f} · '
            f'Teal = positive CLV impact · Red = negative CLV impact</sup>'
        ),
        x=0.5,
        font=dict(size=14),
    ),
    xaxis=dict(
        title='% Change in Median Portfolio CLV vs Baseline',
        zeroline=True,
        zerolinewidth=2.5,
        zerolinecolor='#2C3E50',
        ticksuffix='%',
    ),
    yaxis=dict(autorange='reversed'),
    template='plotly_white',
    height=380,
    legend=dict(
        x=0.60, y=0.08,
        bgcolor='rgba(255,255,255,0.92)',
        bordercolor='#BDC3C7',
        borderwidth=1,
    ),
    margin=dict(l=320, r=110, t=130, b=70),
)
fig_tornado.show()

In [ ]:
# ── Compute CAC thresholds at multiple payback windows ────────────────────────
rm_biz = RiskMetrics(portfolio_results)

cac_18 = rm_biz.optimal_cac(payback_months=18)
cac_24 = rm_biz.optimal_cac(payback_months=24)
cac_36 = rm_biz.optimal_cac(payback_months=36)
cac_48 = rm_biz.optimal_cac(payback_months=48)

portfolio_median_clv = portfolio_results['clv_p50'].median()

# ── Retention ROI from tornado ────────────────────────────────────────────────
cm_result = sensitivity_df[sensitivity_df['parameter'] == 'churn_multiplier'].iloc[0]
retention_upside_pct  = cm_result['change_high_pct']  # better retention
retention_downside_pct = cm_result['change_low_pct']  # worse retention
retention_value_gain  = portfolio_median_clv * (retention_upside_pct / 100)
retention_value_loss  = portfolio_median_clv * (abs(retention_downside_pct) / 100)

dr_result = sensitivity_df[sensitivity_df['parameter'] == 'discount_rate_annual'].iloc[0]

print('=' * 60)
print('BUSINESS METRICS SUMMARY')
print('=' * 60)
print(f'\nPortfolio Median CLV:        R{portfolio_median_clv:>10,.2f}')
print(f'\n--- Maximum Defensible CAC by Payback Window ---')
print(f'  18-month payback:   R{cac_18:>8,.2f}/customer  (growth-mode target)')
print(f'  24-month payback:   R{cac_24:>8,.2f}/customer')
print(f'  36-month payback:   R{cac_36:>8,.2f}/customer  <- recommended baseline')
print(f'  48-month payback:   R{cac_48:>8,.2f}/customer  (patient capital)')
print(f'\n--- Retention Impact (from Tornado Analysis) ---')
print(f'  20% better retention -> median CLV {retention_upside_pct:+.1f}%  (+R{retention_value_gain:,.0f}/customer)')
print(f'  20% worse  retention -> median CLV {retention_downside_pct:+.1f}%  (-R{retention_value_loss:,.0f}/customer)')
print(f'\n--- Discount Rate Impact ---')
print(f'  WACC at 10% (low):   -> CLV {dr_result["change_low_pct"]:+.1f}% vs baseline')
print(f'  WACC at 16% (high):  -> CLV {dr_result["change_high_pct"]:+.1f}% vs baseline')

---
## 8. Business Implications

### 8.1 Optimal Customer Acquisition Cost (CAC)

The **maximum defensible CAC** is derived directly from the median portfolio CLV. The formula is straightforward:

$$\text{Max CAC} = \frac{\text{Median Portfolio CLV}}{\text{Target Payback Period (months)}}$$

The payback period represents the business's risk appetite:
- **18-month payback** suits an aggressive growth phase — the business accepts tighter margins in exchange for faster market share.
- **36-month payback** is the industry standard for subscription telcos — profitable within a typical customer contract cycle.
- **48-month payback** suits mature markets where the business prioritises profitability over growth.

The output from the code cell above gives the exact thresholds for this portfolio. If your current CAC budget **exceeds the 36-month threshold**, you are likely running negative LTV:CAC ratios across the Standard and Churn-Risk segments.

> **Rule of thumb:** A healthy subscription business targets LTV:CAC ≥ 3:1. If the 36-month CAC threshold implies a LTV:CAC below 3, the business needs to either reduce acquisition spend or improve CLV through pricing, cross-sell, or retention.

### 8.2 Retention ROI — The Single Biggest Lever

The tornado chart makes a striking finding visible at a glance: **churn rate is the dominant driver of CLV uncertainty**, outweighing even the cost of capital.

A 20% improvement in retention (customers living 20% longer than the model expects) adds more value per customer than the entire range of plausible discount rate variation. In rand terms — as computed above — this translates to a meaningful uplift per customer in the portfolio median.

This has a direct implication for where the business should invest:

> **1 percentage point invested in reducing churn yields more CLV upside than 1 percentage point invested in lowering the cost of capital.**

The tornado chart quantifies exactly *how much more* — giving the finance team a data-driven basis for prioritising retention programmes over financial engineering.

### 8.3 Three Actionable Recommendations

**Recommendation 1: Differentiate acquisition budgets by segment.**

The key metrics table (Table 3) shows median CLV varying by roughly 10–15x across segments. A uniform per-customer CAC target is therefore both over-generous for Churn Risk customers and overly conservative for High Value customers.

The fix: set segment-specific CAC ceilings using `optimal_cac(payback_months=36)` run separately on each segment's CLV distribution. This alone could improve marketing ROI by redirecting budget from low-CLV acquisitions to high-CLV ones.

**Recommendation 2: Front-load retention investment in the Churn Risk segment.**

The Weibull shape parameter for `churn_risk` customers is $k < 1$ — meaning their churn hazard is *highest in the first few months* and falls thereafter. A customer who survives month 3 is less likely to churn in month 4 than they were in month 1.

This creates a clear intervention window: retention investment is most cost-effective in months 1–3. An onboarding programme, early engagement trigger, or "save" offer targeting customers with low early-tenure engagement should be measurably ROI-positive based on the CLV uplift from extending even a fraction of Churn Risk customers into the Standard segment.

**Recommendation 3: Use the P5–P95 band, not the median, to stress-test acquisition decisions.**

Median CLV is the right input for long-run average acquisition planning. But for individual large deals — a corporate account, a multi-year contract negotiation, a subsidised device offer — the P5 CLV is the relevant floor.

If the acquisition cost exceeds the P5 CLV, the business is making a bet it will lose 5% of the time by design. For a portfolio of 1,000 such deals, that means ~50 guaranteed loss-making acquisitions. The simulation produces these percentile estimates for free; using them to gate large deals is a zero-cost risk management improvement.

---

### Summary

| Finding | Recommendation | Priority |
|---|---|---|
| CLV varies 10–15x across segments | Segment-specific CAC ceilings | High |
| Churn is the #1 CLV driver (tornado) | Front-load retention in months 1–3 | High |
| P5 CLV near zero for Churn Risk | Gate large deals on P5, not median | Medium |
| WACC uncertainty has secondary impact | Discount rate modelling can stay simple | Low |